In [24]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from lightgbm import LGBMRegressor  # <-- Importando o LightGBM

In [25]:
# 1. CARREGAR DADOS
df = pd.read_excel('RPVMM.xlsx','Planilha9')
df['dia_semana'] = df['dia_semana'].astype(object)
formula = "ln_volume ~ ln_preço * (C(semana) + C(dia_semana))"
y_matriz, X_formula = dmatrices(formula, data=df, return_type='dataframe')

X = X_formula.drop(columns=['Intercept'])
y = y_matriz['ln_volume']
# Substitui caracteres proibidos pelo LightGBM nos nomes das colunas
X.columns = (X.columns
             .str.replace(':', '_', regex=False)
             .str.replace('[', '_', regex=False)
             .str.replace(']', '_', regex=False)
             .str.replace('(', '_', regex=False)
             .str.replace(')', '_', regex=False)
             .str.replace(' ', '_', regex=False))

In [26]:
# 2. ALGORITMOS DE SELEÇÃO ESTATÍSTICA (statsmodels)
def forward_selection(X, y, significance_level=0.05):
    initial_features = []
    while True:
        remaining_features = list(set(X.columns) - set(initial_features))
        new_pval = pd.Series(index=remaining_features, dtype=float)
        for new_column in remaining_features:
            model = sm.OLS(y, sm.add_constant(X[initial_features + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < significance_level:
            best_feature = new_pval.idxmin()
            initial_features.append(best_feature)
        else:
            break
    return initial_features

def backward_elimination(X, y, significance_level=0.05):
    features = list(X.columns)
    while len(features) > 0:
        model = sm.OLS(y, sm.add_constant(X[features])).fit()
        p_values = model.pvalues.drop('const')
        max_p_value = p_values.max()
        if max_p_value > significance_level:
            excluded_feature = p_values.idxmax()
            features.remove(excluded_feature)
        else:
            break
    return features

def stepwise_selection(X, y, threshold_in=0.05, threshold_out=0.05):
    included = []
    while True:
        changed = False
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded, dtype=float)
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
        if included:
            model = sm.OLS(y, sm.add_constant(X[included])).fit()
            p_values = model.pvalues.drop('const')
            max_p_value = p_values.max()
            if max_p_value > threshold_out:
                worst_feature = p_values.idxmax()
                included.remove(worst_feature)
                changed = True
        if not changed:
            break
    return included

In [27]:
forward_selection(X, y)
backward_elimination(X, y)
stepwise_selection(X, y)

['C_dia_semana__T.sab_',
 'ln_preço_C_dia_semana__T.sex_',
 'ln_preço',
 'C_dia_semana__T.seg_',
 'ln_preço_C_dia_semana__T.sab_']

In [28]:
# 3. VALIDAÇÃO CRUZADA (5-FOLD) PARA TODOS OS MÉTODOS
kf = KFold(n_splits=5, shuffle=True, random_state=42)

metodos_estatisticos = {
    'Forward': forward_selection, 
    'Backward': backward_elimination, 
    'Stepwise': stepwise_selection
}

# Inicializa o histórico estendido para incluir o LightGBM
historico = {m: {'R2_Treino':[], 'R2_Teste':[], 'RMSE_Treino (kg)':[], 'RMSE_Teste (kg)':[], 'WMAPE_Treino (%)':[], 'WMAPE_Teste (%)':[]} 
             for m in lista_modelos}

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Valores REAIS convertidos de volta para KG usando a função inversa (Exponencial)
    y_train_kg = np.exp(y_train)
    y_test_kg = np.exp(y_test)
    
    # --- PROCESSAMENTO DOS MÉTODOS STATSMODELS ---
    for nome, funcao_selecao in metodos_estatisticos.items():
        vars_sel = funcao_selecao(X_train, y_train)
        if len(vars_sel) == 0:
            pred_train = np.full_like(y_train, np.mean(y_train), dtype=float)
            pred_test = np.full_like(y_test, np.mean(y_train), dtype=float)
            r2_adj = 0.0
        else:
            X_tr_const = sm.add_constant(X_train[vars_sel])
            model = sm.OLS(y_train, X_tr_const).fit()
            pred_train = model.predict(X_tr_const)
            r2_adj = model.rsquared_adj
            X_te_const = sm.add_constant(X_test[vars_sel], has_constant='add')
            pred_test = model.predict(X_te_const)
            
        # Convertendo as PREDIÇÕES de Log para KG
        pred_train_kg = np.exp(pred_train)
        pred_test_kg = np.exp(pred_test)
            
        historico[nome]['R2_Treino'].append(r2_adj)
        historico[nome]['R2_Teste'].append(r2_score(y_test, pred_test)) # R² calculado na escala original do modelo (log)
        historico[nome]['RMSE_Treino (kg)'].append(np.sqrt(mean_squared_error(y_train_kg, pred_train_kg)))
        historico[nome]['RMSE_Teste (kg)'].append(np.sqrt(mean_squared_error(y_test_kg, pred_test_kg)))
        historico[nome]['WMAPE_Treino (%)'].append((np.sum(np.abs(y_train_kg - pred_train_kg)) / np.sum(y_train_kg)) * 100)
        historico[nome]['WMAPE_Teste (%)'].append((np.sum(np.abs(y_test_kg - pred_test_kg)) / np.sum(y_test_kg)) * 100)
        
        
    # --- PROCESSAMENTO DO LASSO ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000, tol=0.01)
    lasso_cv.fit(X_train_scaled, y_train)
    
    modelo_lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=100000, tol=0.01)
    modelo_lasso.fit(X_train_scaled, y_train)
    
    pred_train_lasso_kg = np.exp(modelo_lasso.predict(X_train_scaled))
    pred_test_lasso_kg = np.exp(modelo_lasso.predict(X_test_scaled))
    
    historico['Lasso']['R2_Treino'].append(r2_score(y_train, modelo_lasso.predict(X_train_scaled)))
    historico['Lasso']['R2_Teste'].append(r2_score(y_test, modelo_lasso.predict(X_test_scaled)))
    historico['Lasso']['RMSE_Treino (kg)'].append(np.sqrt(mean_squared_error(y_train_kg, pred_train_lasso_kg)))
    historico['Lasso']['RMSE_Teste (kg)'].append(np.sqrt(mean_squared_error(y_test_kg, pred_test_lasso_kg)))
    historico['Lasso']['WMAPE_Treino (%)'].append((np.sum(np.abs(y_train_kg - pred_train_lasso_kg)) / np.sum(y_train_kg)) * 100)
    historico['Lasso']['WMAPE_Teste (%)'].append((np.sum(np.abs(y_test_kg - pred_test_lasso_kg)) / np.sum(y_test_kg)) * 100)


    # --- PROCESSAMENTO DO LIGHTGBM ---
    # verbose=-1 desativa os logs internos de texto do LightGBM para manter a tela limpa
    modelo_lgbm = LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
    modelo_lgbm.fit(X_train, y_train)
    
    pred_train_lgb_kg = np.exp(modelo_lgbm.predict(X_train))
    pred_test_lgb_kg = np.exp(modelo_lgbm.predict(X_test))
    
    historico['LightGBM']['R2_Treino'].append(r2_score(y_train, modelo_lgbm.predict(X_train)))
    historico['LightGBM']['R2_Teste'].append(r2_score(y_test, modelo_lgbm.predict(X_test)))
    historico['LightGBM']['RMSE_Treino (kg)'].append(np.sqrt(mean_squared_error(y_train_kg, pred_train_lgb_kg)))
    historico['LightGBM']['RMSE_Teste (kg)'].append(np.sqrt(mean_squared_error(y_test_kg, pred_test_lgb_kg)))
    historico['LightGBM']['WMAPE_Treino (%)'].append((np.sum(np.abs(y_train_kg - pred_train_lgb_kg)) / np.sum(y_train_kg)) * 100)
    historico['LightGBM']['WMAPE_Teste (%)'].append((np.sum(np.abs(y_test_kg - pred_test_lgb_kg)) / np.sum(y_test_kg)) * 100)


In [29]:
# 4. EXIBIÇÃO DA TABELA COMPARATIVA FINAL
resumos = {m: {k: np.mean(v) for k, v in historico[m].items()} for m in historico}
df_comparativo_final = pd.DataFrame(resumos).T

colunas_ordenadas = ['R2_Treino', 'R2_Teste', 'RMSE_Treino (kg)', 'RMSE_Teste (kg)', 'WMAPE_Treino (%)', 'WMAPE_Teste (%)']
print("\n======================= TABELA COMPARATIVA FINAL COM LIGHTGBM =======================")
print(df_comparativo_final[colunas_ordenadas].round(4).to_string())
print("======================================================================================")


======================= TABELA COMPARATIVA FINAL COM LIGHTGBM =======================
          R2_Treino  R2_Teste  RMSE_Treino (kg)  RMSE_Teste (kg)  WMAPE_Treino (%)  WMAPE_Teste (%)
Forward      0.6416    0.2251           11.3858          12.8322           35.0858          44.7753
Backward     0.8714   -1.4066            9.4578      213993.7496           27.3134      259808.5722
Stepwise     0.6307    0.2195           11.5673          13.0882           35.5842          45.2767
Lasso        0.8495    0.6934            9.9916          11.0998           29.4613          36.2291
LightGBM     0.0571   -0.3108           19.5158          19.3606           64.3948          69.4935


In [32]:
# 5. TREINAMENTO DO LASSO NA BASE COMPLETA E EXTRAÇÃO DE COEFICIENTES EM KG
print("\n======================= AJUSTANDO MODELO LASSO FINAL (BASE TOTAL) =======================")
scaler_final = StandardScaler()
X_scaled_total = scaler_final.fit_transform(X)

# Otimização final do alpha na base inteira
lasso_cv_final = LassoCV(cv=5, random_state=42, max_iter=100000, tol=0.01)
lasso_cv_final.fit(X_scaled_total, y)

modelo_lasso_final = Lasso(alpha=lasso_cv_final.alpha_, max_iter=100000, tol=0.01)
modelo_lasso_final.fit(X_scaled_total, y)

# Estruturando os coeficientes obtidos
df_coefs = pd.DataFrame({
    'Variável': X.columns,
    'Coef_Log (Beta)': modelo_lasso_final.coef_
})
df_coefs['Impacto_Multiplicativo (e^Beta)'] = np.exp(df_coefs['Coef_Log (Beta)'])
mantidas = df_coefs[df_coefs['Coef_Log (Beta)'] != 0].sort_values(by='Coef_Log (Beta)', ascending=False)
eliminadas = df_coefs[df_coefs['Coef_Log (Beta)'] == 0]['Variável'].tolist()
print(f"Intercepto Final (Escala Log): {modelo_lasso_final.intercept_.round(4)}")
print(f"Intercepto Final convertido (Demanda Base em kg): {np.exp(modelo_lasso_final.intercept_).round(2)} kg\n")
print("--- VARIÁVEIS MANTIDAS PELO LASSO E IMPACTO REAL EM KG ---")
print(mantidas.to_string(index=False))
print(f"\n--- VARIÁVEIS ELIMINADAS PELO LASSO (ZURADAS): {len(eliminadas)} ---")
print(eliminadas)


======================= AJUSTANDO MODELO LASSO FINAL (BASE TOTAL) =======================
Intercepto Final (Escala Log): 2.5436
Intercepto Final convertido (Demanda Base em kg): 12.73 kg

--- VARIÁVEIS MANTIDAS PELO LASSO E IMPACTO REAL EM KG ---
                     Variável  Coef_Log (Beta)  Impacto_Multiplicativo (e^Beta)
         C_dia_semana__T.seg_         0.842065                         2.321156
         C_dia_semana__T.ter_         0.722842                         2.060280
         C_dia_semana__T.qui_         0.687672                         1.989079
         C_dia_semana__T.qua_         0.657817                         1.930573
ln_preço_C_dia_semana__T.qua_         0.509849                         1.665039
ln_preço_C_dia_semana__T.sex_         0.474266                         1.606834
ln_preço_C_dia_semana__T.qui_         0.460226                         1.584432
ln_preço_C_dia_semana__T.seg_         0.436560                         1.547375
ln_preço_C_dia_semana__T.ter_   